<a href="https://colab.research.google.com/github/SakshiIshwe0604/langchain-practice/blob/main/tool_calling/tool_calling_in_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from getpass import getpass
import os

os.environ["GROQ_API_KEY"] = getpass("Enter Groq API key: ")

Enter Groq API key: ··········


In [4]:
!pip install -q langchain-openai langchain-core requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 13.0 MB/s eta 0:00:00


In [7]:
!pip install langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.3 MB/s eta 0:00:00


In [8]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [9]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [10]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [11]:
multiply.name

'multiply'

In [12]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [13]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [ ]:
# tool binding

In [15]:
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [16]:
llm.invoke('hi')

AIMessage(content="It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 36, 'total_tokens': 59, 'completion_time': 0.044528432, 'completion_tokens_details': None, 'prompt_time': 0.004717034, 'prompt_tokens_details': None, 'queue_time': 0.008565014, 'total_time': 0.049245466}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e172c-2e5e-70f1-8d7b-0d09b52aea4e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 23, 'total_tokens': 59})

In [17]:
llm_with_tools = llm.bind_tools([multiply])

In [18]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="I'm doing well, thanks for asking. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 234, 'total_tokens': 259, 'completion_time': 0.088272354, 'completion_tokens_details': None, 'prompt_time': 0.019440807, 'prompt_tokens_details': None, 'queue_time': 0.137145498, 'total_time': 0.107713161}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e1730-bfb8-7c92-8a57-86e873db9c5f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 234, 'output_tokens': 25, 'total_tokens': 259})

In [19]:
query = HumanMessage('can you multiply 3 with 1000')

In [20]:
messages = [query]

In [21]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [22]:
result = llm_with_tools.invoke(messages)

In [23]:
messages.append(result)

In [24]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'wmsf0dp32', 'function': {'arguments': '{"a":3,"b":1000}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 239, 'total_tokens': 259, 'completion_time': 0.068503571, 'completion_tokens_details': None, 'prompt_time': 0.013492992, 'prompt_tokens_details': None, 'queue_time': 0.089014226, 'total_time': 0.081996563}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_0761e44d7b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e1730-f193-7493-a4b1-2801d453baa9-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'wmsf0dp32', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 239, 'output_tokens': 20, 'total_tokens': 259})]

In [25]:
tool_result = multiply.invoke(result.tool_calls[0])

In [26]:
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='wmsf0dp32')

In [27]:
messages.append(tool_result)

In [28]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'wmsf0dp32', 'function': {'arguments': '{"a":3,"b":1000}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 239, 'total_tokens': 259, 'completion_time': 0.068503571, 'completion_tokens_details': None, 'prompt_time': 0.013492992, 'prompt_tokens_details': None, 'queue_time': 0.089014226, 'total_time': 0.081996563}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_0761e44d7b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e1730-f193-7493-a4b1-2801d453baa9-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'wmsf0dp32', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 239, 'output_tokens': 20, 'total_tokens': 259}),
 Too

In [29]:
llm_with_tools.invoke(messages).content

'The result of multiplying 3 by 1000 is 3000.'

In [30]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [31]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [32]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1778457602,
 'time_last_update_utc': 'Mon, 11 May 2026 00:00:02 +0000',
 'time_next_update_unix': 1778544002,
 'time_next_update_utc': 'Tue, 12 May 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.57}

In [ ]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [33]:
# tool binding
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [34]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [35]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [36]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [37]:
ai_message = llm_with_tools.invoke(messages)

In [38]:
messages.append(ai_message)

In [39]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': '4253gz35t',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': '75vwtwnr9',
  'type': 'tool_call'}]

In [40]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [41]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '4253gz35t', 'function': {'arguments': '{"base_currency":"INR","target_currency":"USD"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': '75vwtwnr9', 'function': {'arguments': '{"base_currency_value":10}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 350, 'total_tokens': 388, 'completion_time': 0.073688647, 'completion_tokens_details': None, 'prompt_time': 0.089953229, 'prompt_tokens_details': None, 'queue_time': 0.15395079, 'total_time': 0.163641876}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ce7bc1685b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e1740-d7da-7d53-9b50

In [42]:
llm_with_tools.invoke(messages).content

'The conversion factor between INR and USD is 0.0133. Based on this conversion factor, 10 INR is equivalent to 0.133 USD.'

In [47]:
!pip install -U langchain langchain-community langchain-core -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [49]:
import langchain
print(langchain.__version__)

1.2.15


In [52]:
!pip install langchain==0.3.25 langchain-community==0.3.23 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 65.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-groq 1.1.2 requires langchain-core<2.0.0,>=1.2.8, but you have langchain-core 0.3.86 which is incompatible.
langgraph 1.1.10 requires langchain-core<2,>=1.3.0, but you have langchain-core 0.3.86 which is incompatible.
langchain-openai 1.2.1 requires langchain-core<2.0.0,>=1.3.2, but you have langchain-core 0.3.86 which is incompatible.
langgr

In [59]:
!pip install langchain==0.3.7 langchain-community==0.3.7 langchain-groq -q

In [61]:
import subprocess
subprocess.run(["pip", "install", "langchain==0.3.7", "langchain-community==0.3.7", "--force-reinstall", "-q"])

import importlib
import langchain
importlib.reload(langchain)

<module 'langchain' from '/usr/local/lib/python3.12/dist-packages/langchain/__init__.py'>

In [60]:
from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents import initialize_agent, AgentType

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)

@tool
def get_conversion_factor(unit_from: str, unit_to: str) -> float:
    """Get conversion factor between two units"""
    # your code here
    pass

@tool
def convert(value: float, conversion_factor: float) -> float:
    """Convert a value using conversion factor"""
    return value * conversion_factor

agent_executor = initialize_agent(
    tools=[get_conversion_factor, convert],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)


ImportError: cannot import name 'initialize_agent' from 'langchain.agents' (/usr/local/lib/python3.12/dist-packages/langchain/agents/__init__.py)

In [54]:
!pip show langchain | grep Version

Version: 0.3.25


In [ ]:
# --- Step 6: Run the Agent ---
user_query = "Hi how are you?"

response = agent_executor.invoke({"input": user_query})



> Entering new AgentExecutor chain...
I'm here and ready to help! How can I assist you today?

> Finished chain.
